# NHS England Maternal Care Pathways Master Pipeline
## Stage 7.2 - Define preliminary pre-eclampsia diagnosis

### Compiled by Federica Caretta Cortegiani

In [0]:
import sys
import numpy as np
import pandas as pd
from scipy.stats import norm
import matplotlib.pyplot as plt
from functools import reduce
from pyspark.sql import functions as F
from pyspark.sql import SparkSession 
from pyspark.sql.functions import when, col, lit, count, max, last, first, min, avg, sum, collect_list, collect_set, countDistinct, to_date, datediff, array_contains, size, udf, format_number, regexp_replace, explode, array, array_union, desc_nulls_last, flatten, split, filter
from pyspark.sql.types import IntegerType, StringType, NumericType, DateType, ArrayType
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, Imputer
from pyspark.ml.functions import vector_to_array
from pyspark.sql.window import Window
from pyspark.ml.regression import GeneralizedLinearRegression as GLR

%run "/Workspace/Shared/Helper_Methods_EP"

In [0]:
spark = SparkSession.builder.getOrCreate()

In [0]:
read_table_name = "msds_diag_busy_filtered_final_imputed_stage"

df_master = spark.read.table(f"dsa_712819_x8g2j.dsa_712819_x8g2j_collab.{read_table_name}")

In [0]:
df_master = (
    df_master
    .withColumn(
        "est_pregnancy_start",
            when(col("est_delivery_date").isNotNull(), F.date_add(col("est_delivery_date"), -40*7))
            .otherwise(None)
    )
    .withColumn("gestation_length", F.date_diff("ld_hosp_disch_date", "last_period_date"))
    .withColumn(
        "ld_hosp_disch_date", when(((col("gestation_length") > 42*7) | (col("gestation_length") < 0)), F.date_add(col("last_period_date"), 42*7)).otherwise(col("ld_hosp_disch_date"))
    )
    .drop(*["gestation_length"])
)

In [0]:
df_master_pregs = (
    df_master
    .select("UniqPregID", "Person_ID_DEID", "last_period_date", "ld_hosp_start_date", "ld_hosp_disch_date")
)

In [0]:
ECDS = 1
MSDS = 1
APC  = 1
AE   = 0

In [0]:
preeclampsia_codes = [
    # ICD-10
    "O14", "O14.0", "O14.1", "O14.2", "O14.9", "O11", "O15", "O15.0", "O15.1", "O15.2", "O15.9",
    "O10", "O13", "O16",
    # SNOMED
    "15938005", "29738008", "71701000119105", "72022006", "198992004", "169560008",
    "398254007", "427889009", "48194001", "69909000", "111479006"
]

gestational_diabetes_codes = [
    # ICD-10
    "O24.4", "O24.410", "O24.414", "O24.419", "O24.42", "O24.43", "O24.9"
    
    # SNOMED
    "11687002", "73211009", "724714000", "724715004", "724716003"
]

In [0]:
df_ecds = spark.sql("SELECT DISTINCT PERSON_ID_DEID, fyear, ARRIVAL_DATE, COMORBIDITIES_1, COMORBIDITIES_2, COMORBIDITIES_3, COMORBIDITIES_4, COMORBIDITIES_5, COMORBIDITIES_6, COMORBIDITIES_7, COMORBIDITIES_8, COMORBIDITIES_9, COMORBIDITIES_10, DIAGNOSIS_CODE_1, DIAGNOSIS_CODE_2, DIAGNOSIS_CODE_3, DIAGNOSIS_CODE_4, DIAGNOSIS_CODE_5, DIAGNOSIS_CODE_6, DIAGNOSIS_CODE_7, DIAGNOSIS_CODE_8, DIAGNOSIS_CODE_9, DIAGNOSIS_CODE_10 FROM `dsa_712819_x8g2j`.`dsa_712819_x8g2j_collab`.`lowlat_ecds_all_years`;")

In [0]:
# Join A&E data to pregnancies

df_ecds_diags = df_ecds.join(df_master_pregs, on="Person_ID_DEID", how="inner")

df_ecds_diags = df_ecds_diags.filter((col("Arrival_Date") >= col("last_period_date")) & (col("Arrival_Date") <= col("ld_hosp_disch_date")))

df_ecds_diags = (
    df_ecds_diags 
    .groupBy("UniqPregID", "ARRIVAL_DATE")
    .agg(
        * [collect_set(c).alias(c) for c in df_ecds_diags.columns if c.startswith(("DIAGNOSIS_CODE_", "COMORBIDITIES_"))]
    )
)

# Collect all diagnosis and comorbidity cols 
diagnosis_cols = [f"DIAGNOSIS_CODE_{i}" for i in range(1, 11)]
comorbidity_cols = [f"COMORBIDITIES_{i}" for i in range(1, 11)]

diagnosis_union = col(diagnosis_cols[0])
for col_name in diagnosis_cols[1:]:
    diagnosis_union = array_union(diagnosis_union, col(col_name))

comorbidity_union = col(comorbidity_cols[0])
for col_name in comorbidity_cols[1:]:
    comorbidity_union = array_union(comorbidity_union, col(col_name))

df_ecds_diags = df_ecds_diags.withColumns({
    f"DiagnosisCodes": diagnosis_union,
    f"Comorbidities": comorbidity_union,
    "DiagnosisDate": col("ARRIVAL_DATE")
}).select(
    "UniqPregID", "DiagnosisDate", "DiagnosisCodes", "Comorbidities"
)


In [0]:
ecds_diag_desc = spark.table("dsa_712819_x8g2j.dsa_712819_x8g2j_collab.ecdsdescriptionsdiag")
ecds_diag_desc= ecds_diag_desc.select(
    col("diagnosis_code_snomed").cast("string").alias("diag_code"),
    col("higher_level_description").alias("diag_category")
)

ecds_diag_desc = ecds_diag_desc.withColumn(
    "diag_code",
    regexp_replace(col("diag_code").cast("string"), r"\.0$", "")
).select(
    "diag_code",
    "diag_category"
)

# Collect lookup as dict
code_to_desc = {row['diag_code']: row['diag_category'] for row in ecds_diag_desc.collect()}

# Define UDF
def map_codes_to_desc(codes):
    if codes is None:
        return None
    return [code_to_desc.get(c, f"Unknown({c})") for c in codes]

map_codes_udf = udf(map_codes_to_desc, ArrayType(StringType()))

# Apply UDF
df_ecds_diags = df_ecds_diags.withColumn(
    "DiagnosisDescriptions",
    map_codes_udf("DiagnosisCodes")
)

In [0]:
ecds_comorb_desc = spark.table("dsa_712819_x8g2j.dsa_712819_x8g2j_collab.ecdsdescriptionscomorb")
ecds_comorb_desc= ecds_comorb_desc.select(
    col("comorbidity_code").cast("string").alias("comorb_code"),
    col("higher_level_description").alias("comorb_category")
)

ecds_comorb_desc = ecds_comorb_desc.withColumn(
    "comorb_code",
    regexp_replace(col("comorb_code").cast("string"), r"\.0$", "")
).select(
    "comorb_code",
    "comorb_category"
)

# Convert lookup table to dict
comorb_to_desc = {row['comorb_code']: row['comorb_category'] for row in ecds_comorb_desc.collect()}

# Define UDF
def map_comorbs_to_desc(codes):
    if codes is None:
        return None
    return [comorb_to_desc.get(c, f"Unknown({c})") for c in codes]

map_comorbs_udf = udf(map_comorbs_to_desc, ArrayType(StringType()))

# Apply UDF
df_ecds_diags = df_ecds_diags.withColumn(
    "ComorbidityDescriptions",
    map_comorbs_udf("Comorbidities")
)

In [0]:
df_ecds_diags = df_ecds_diags.withColumns({
    "DiagnosisCodes_ECDS": col("DiagnosisCodes"),
    "DiagnosisDescriptions_ECDS": col("DiagnosisDescriptions"),
    "Comorbidities_ECDS": col("Comorbidities"),
    "ComorbidityDescriptions_ECDS": col("ComorbidityDescriptions")
}).select(
    "UniqPregID", 
    "DiagnosisDate",
    "DiagnosisCodes_ECDS", "DiagnosisDescriptions_ECDS",
    "Comorbidities_ECDS", "ComorbidityDescriptions_ECDS"
)

In [0]:
df_msds_diags = spark.sql(
    """
    SELECT DISTINCT person_id_mother_deid, diagnosisandhistory, diagnosisdate,mastericd10diagnosisandhistorycode, mastericd10diagnosisandhistorydesc, mastersnomedctdiagnosisandhistorycode, mastersnomedctdiagnosisandhistoryterm, diagnosisscheme FROM `dsa_712819_x8g2j`.`dsa_712819_x8g2j_collab`.`msds_v2_diagnoses_and_history_all_years`;
    """
)

df_msds_diags = df_msds_diags.select(
    "person_id_mother_deid",
    "diagnosisandhistory",
    "diagnosisdate"
).withColumnRenamed("person_id_mother_deid", "person_id_deid")

# filter for patients in df_master
df_msds_diags = df_msds_diags.join(
    df_master_pregs, on="person_id_deid", how="inner"
).filter(
    (col("diagnosisdate") <= col("ld_hosp_disch_date")) & \
    (col("diagnosisdate") >= col("last_period_date"))
).groupBy("UniqPregID", "diagnosisdate").agg(
    collect_set("diagnosisandhistory").alias("DiagnosisCodes_MSDS")
)

In [0]:
df_msdsdesc = spark.sql("""
    SELECT DISTINCT 
        diagnosis_code_snomed,
        icd10,
        higher_level_description
    FROM dsa_712819_x8g2j_collab.msdsdescriptions
""")

# Update `higher_level_description` based on match
df_msdsdesc_updated = df_msdsdesc.withColumn(
    "higher_level_description",
    when(col("icd10").isin(preeclampsia_codes) | col("diagnosis_code_snomed").isin(preeclampsia_codes), "Preeclampsia")
    .when((col("icd10").isin(gestational_diabetes_codes)) | (col("diagnosis_code_snomed").isin(gestational_diabetes_codes)), "Gestational Diabetes")
    .otherwise(col("higher_level_description"))
)

In [0]:
# Collect mapping of codes to descriptions as dict
msds_desc_rows = df_msdsdesc_updated.select("diagnosis_code_snomed", "icd10", "higher_level_description").collect()

code_to_desc = {}
for row in msds_desc_rows:
    if row["diagnosis_code_snomed"]:
        code_to_desc[row["diagnosis_code_snomed"]] = row["higher_level_description"]
    if row["icd10"]:
        code_to_desc[row["icd10"]] = row["higher_level_description"]

# Define UDF
def map_msds_codes_to_desc(codes):
    if codes is None:
        return None
    return [code_to_desc.get(c, f"Unknown({c})") for c in codes]

map_msds_udf = udf(map_msds_codes_to_desc, ArrayType(StringType()))

# Apply UDF
df_msds_diags = df_msds_diags.withColumn(
    "DiagnosisDescriptions_MSDS",
    map_msds_udf("DiagnosisCodes_MSDS")
).select(
    "UniqPregID", "DiagnosisDate",
    "DiagnosisCodes_MSDS", "DiagnosisDescriptions_MSDS"
)

In [0]:
# Load hosp diagnosis data
df_apc_diags = spark.sql(
"""
    SELECT PERSON_ID_DEID, ADMIDATE, DIAG_3_CONCAT, DIAG_4_CONCAT
    FROM dsa_712819_x8g2j.dsa_712819_x8g2j_collab.hes_apc_all_years
"""
)

# Join to pregnancies in df_master and filter for timing before lmp
df_apc_diags = df_apc_diags.join(
    df_master_pregs,
    on="Person_ID_DEID",
    how="inner"
)

df_apc_diags = df_apc_diags.filter((col("ADMIDATE") <= col("ld_hosp_disch_date")) & (col("ADMIDATE") >= col("last_period_date"))).withColumnRenamed("ADMIDATE", "DiagnosisDate")


In [0]:
# Group and collect diagnosis lists
df_apc_diags = (
    df_apc_diags
    .groupBy("UniqPregID", "DiagnosisDate") 
    .agg(
        collect_list("DIAG_3_CONCAT").alias(f"Diag3Codes_APC"),
        collect_list("DIAG_4_CONCAT").alias(f"Diag4Codes_APC")
    )
)

df_apc_diags = df_apc_diags.withColumn(
    f"Diag3Codes_APC",
    F.array_distinct(
        F.filter(
            F.flatten(
                F.transform(
                    col(f"Diag3Codes_APC"),
                    lambda x: F.split(x, ",")
                )
            ),
            lambda y: y != ""
        )
    )
).withColumn(
    f"Diag4Codes_APC",
    F.array_distinct(
        F.filter(
            F.flatten(
                F.transform(
                    col(f"Diag4Codes_APC"),
                    lambda x: F.split(x, ",")
                )
            ),
            lambda y: y != ""
        )
    )
)

In [0]:
df_first = spark.sql("SELECT * FROM dsa_712819_x8g2j_collab.apcdescriptionsfirst_cleaned")
df_second = spark.sql("SELECT * FROM dsa_712819_x8g2j_collab.apcdescriptionssecond")

apc_desc_dict = {}

def extract_code_mappings(row, df_cols):
    for col in df_cols:
        if col.startswith("code"):
            icd_agg_col = col.replace("code", "icd_agg_desc")
            if icd_agg_col in df_cols:
                code_val = row[col]
                agg_desc_val = row[icd_agg_col]
                if code_val and agg_desc_val:
                    apc_desc_dict[code_val] = agg_desc_val

first_cols = df_first.columns
for row in df_first.collect():
    extract_code_mappings(row, first_cols)

second_cols = df_second.columns
for row in df_second.collect():
    extract_code_mappings(row, second_cols)

# Define mapping function 
def map_to_agg_desc(codes):
    if codes is None:
        return None
    result = []
    for c in codes:
        if c in preeclampsia_codes:
            result.append("Preeclampsia")
        elif c in gestational_diabetes_codes:
            result.append("Gestational Diabetes")
        else:
            result.append(apc_desc_dict.get(c, f"Unknown({c})"))
    return result

map_agg_desc_udf = udf(map_to_agg_desc, ArrayType(StringType()))

df_apc_diags = df_apc_diags.withColumns({
    "Diag3AggDescriptions_APC": map_agg_desc_udf("Diag3Codes_APC"),
    "Diag4AggDescriptions_APC": map_agg_desc_udf("Diag4Codes_APC")
}).select(
    "UniqPregID", "DiagnosisDate",
    "Diag3Codes_APC", "Diag3AggDescriptions_APC",
    "Diag4Codes_APC", "Diag4AggDescriptions_APC"
)


In [0]:
# Define columns
diag2_cols = [
    "DIAG2_01", "DIAG2_02", "DIAG2_03", "DIAG2_04",
    "DIAG2_05", "DIAG2_06", "DIAG2_07", "DIAG2_08",
    "DIAG2_09", "DIAG2_10", "DIAG2_11", "DIAG2_12"
]
diag3_cols = [
    "DIAG3_01", "DIAG3_02", "DIAG3_03", "DIAG3_04",
    "DIAG3_05", "DIAG3_06", "DIAG3_07", "DIAG3_08",
    "DIAG3_09", "DIAG3_10", "DIAG3_11", "DIAG3_12"
]
diag_ae_cols = ["PERSON_ID_DEID", "ARRIVALDATE"] + diag2_cols + diag3_cols

# Load A&E data
df_hes_ae = spark.sql("""
    SELECT *
    FROM dsa_712819_x8g2j.dsa_712819_x8g2j_collab.hes_ae_all_years
""")
df_ae_diags = df_hes_ae.select(*diag_ae_cols)

# Join to master preg list
df_ae_diags = df_ae_diags.join(
    df_master_pregs,
    on="Person_ID_DEID",
    how="inner"
)

df_ae_diags = df_ae_diags.filter(
    (col("ARRIVALDATE") <= col("ld_hosp_disch_date")) & \
    (col("ARRIVALDATE") >= col("last_period_date"))
)

In [0]:
# Group and collect diagnosis lists
df_ae_diags = (
    df_ae_diags
    .groupBy("UniqPregID", "ARRIVALDATE")
    .agg(
        collect_list("DIAG2_01").alias(f"Diag2Codes_01_AE"),
        collect_list("DIAG2_02").alias(f"Diag2Codes_02_AE"),
        collect_list("DIAG2_03").alias(f"Diag2Codes_03_AE"),
        collect_list("DIAG2_04").alias(f"Diag2Codes_04_AE"),
        collect_list("DIAG2_05").alias(f"Diag2Codes_05_AE"),
        collect_list("DIAG2_06").alias(f"Diag2Codes_06_AE"),
        collect_list("DIAG2_07").alias(f"Diag2Codes_07_AE"),
        collect_list("DIAG2_08").alias(f"Diag2Codes_08_AE"),
        collect_list("DIAG2_09").alias(f"Diag2Codes_09_AE"),
        collect_list("DIAG2_10").alias(f"Diag2Codes_10_AE"),
        collect_list("DIAG2_11").alias(f"Diag2Codes_11_AE"),
        collect_list("DIAG2_12").alias(f"Diag2Codes_12_AE"),

        collect_list("DIAG3_01").alias(f"Diag3Codes_01_AE"),
        collect_list("DIAG3_02").alias(f"Diag3Codes_02_AE"),
        collect_list("DIAG3_03").alias(f"Diag3Codes_03_AE"),
        collect_list("DIAG3_04").alias(f"Diag3Codes_04_AE"),
        collect_list("DIAG3_05").alias(f"Diag3Codes_05_AE"),
        collect_list("DIAG3_06").alias(f"Diag3Codes_06_AE"),
        collect_list("DIAG3_07").alias(f"Diag3Codes_07_AE"),
        collect_list("DIAG3_08").alias(f"Diag3Codes_08_AE"),
        collect_list("DIAG3_09").alias(f"Diag3Codes_09_AE"),
        collect_list("DIAG3_10").alias(f"Diag3Codes_10_AE"),
        collect_list("DIAG3_11").alias(f"Diag3Codes_11_AE"),
        collect_list("DIAG3_12").alias(f"Diag3Codes_12_AE")
    ) 
    .withColumns({
        f"Diag2Codes_AE": F.array_distinct(
            F.flatten(F.array(
                col(f"Diag2Codes_01_AE"),
                col(f"Diag2Codes_02_AE"),
                col(f"Diag2Codes_03_AE"),
                col(f"Diag2Codes_04_AE"),
                col(f"Diag2Codes_05_AE"),
                col(f"Diag2Codes_06_AE"),
                col(f"Diag2Codes_07_AE"),
                col(f"Diag2Codes_08_AE"),
                col(f"Diag2Codes_09_AE"),
                col(f"Diag2Codes_10_AE"),
                col(f"Diag2Codes_11_AE"),
                col(f"Diag2Codes_12_AE")
            ))
        ),
        f"Diag3Codes_AE": F.array_distinct(
            F.flatten(F.array(
                col(f"Diag3Codes_01_AE"),
                col(f"Diag3Codes_02_AE"),
                col(f"Diag3Codes_03_AE"),
                col(f"Diag3Codes_04_AE"),
                col(f"Diag3Codes_05_AE"),
                col(f"Diag3Codes_06_AE"),
                col(f"Diag3Codes_07_AE"),
                col(f"Diag3Codes_08_AE"),
                col(f"Diag3Codes_09_AE"),
                col(f"Diag3Codes_10_AE"),
                col(f"Diag3Codes_11_AE"),
                col(f"Diag3Codes_12_AE")
            ))
        )
    }) 
    .select("UniqPregID", "ARRIVALDATE", f"Diag2Codes_AE", f"Diag3Codes_AE")
)



In [0]:
# Load lookup
df_lookup = spark.sql("SELECT * FROM dsa_712819_x8g2j_collab.aedescriptions")

# Create dictionary
lookup_dict = {}

for row in df_lookup.collect():
    code_val = row['code']
    desc_val = row['higher_level_description']
    if code_val and desc_val:
        lookup_dict[str(code_val)] = desc_val

# Define UDF
def map_ae_codes(code_list):
    if code_list is None:
        return None
    return [lookup_dict.get(str(c), f"Unknown({c})") for c in code_list]

map_ae_udf = udf(map_ae_codes, ArrayType(StringType()))

# Apply and add cols
df_ae_diags = df_ae_diags.withColumns({
    "Diag2Descriptions_AE": map_ae_udf("Diag2Codes_AE"),
    "Diag3Descriptions_AE": map_ae_udf("Diag3Codes_AE")
}).select(
    "UniqPregID", "ARRIVALDATE",
    "Diag2Codes_AE", "Diag2Descriptions_AE",
    "Diag3Codes_AE", "Diag3Descriptions_AE"
).withColumnRenamed("ARRIVALDATE", "DiagnosisDate")

In [0]:
keys = ["UniqPregID", "DiagnosisDate"]

df_ecds_diags = df_ecds_diags.select(*keys, "DiagnosisDescriptions_ECDS", "ComorbidityDescriptions_ECDS")
df_msds_diags = df_msds_diags.select(*keys, "DiagnosisDescriptions_MSDs")
df_apc_diags  = df_apc_diags.select(*keys, "Diag3AggDescriptions_APC", "Diag4AggDescriptions_APC")
df_ae_diags   = df_ae_diags.select(*keys, "Diag2Descriptions_AE", "Diag3Descriptions_AE")

df_all_diags = None

if ECDS == 1:
  df_all_diags = df_ecds_diags

if MSDS == 1:
  df_all_diags = df_all_diags.join(df_msds_diags, ["UniqPregID", "DiagnosisDate"], "outer") if df_all_diags is not None else df_msds_diags

if APC == 1:
  df_all_diags = df_all_diags.join(df_apc_diags, ["UniqPregID", "DiagnosisDate"], "outer") if df_all_diags is not None else df_apc_diags

if AE == 1:
  df_all_diags = df_all_diags.join(df_ae_diags, ["UniqPregID", "DiagnosisDate"], "outer") if df_all_diags is not None else df_ae_diags

parts = []

if ECDS == 1:
  parts.append(F.coalesce(F.col("DiagnosisDescriptions_ECDS"), F.array(F.lit("missing"))))
  parts.append(F.coalesce(F.col("ComorbidityDescriptions_ECDS"), F.array(F.lit("missing"))))

if MSDS == 1:
  parts.append(F.coalesce(F.col("DiagnosisDescriptions_MSDS"), F.array(F.lit("missing"))))

if APC == 1:
  parts.append(F.coalesce(F.col("Diag3AggDescriptions_APC"), F.array(F.lit("missing"))))
  parts.append(F.coalesce(F.col("Diag4AggDescriptions_APC"), F.array(F.lit("missing"))))

if AE == 1:
  parts.append(F.coalesce(F.col("Diag2Descriptions_AE"), F.array(F.lit("missing"))))
  parts.append(F.coalesce(F.col("Diag3Descriptions_AE"), F.array(F.lit("missing"))))

df_all_diags = (
  df_all_diags
  .withColumn(
    "DiagDescriptions_ALL", F.concat(*parts)
  ).withColumn(
    "DiagDescriptions_ALL", F.array_distinct(col("DiagDescriptions_ALL"))
  ).select(
    "UniqPregID", "DiagnosisDate", "DiagDescriptions_ALL"
  )
)

In [0]:
# List of categories
categories = [
    "Preeclampsia",
    "Gestational Diabetes",
    "Certain infectious and parasitic diseases",
    "Neoplasms",
    "Diseases of the blood and blood-forming organs and certain disorders involving the immune mechanism",
    "Endocrine, nutritional and metabolic diseases",
    "Mental and behavioural disorders",
    "Diseases of the nervous system",
    "Diseases of the eye and adnexa",
    "Diseases of the ear and mastoid process",
    "Diseases of the circulatory system",
    "Diseases of the respiratory system",
    "Diseases of the digestive system",
    "Diseases of the skin and subcutaneous tissue",
    "Diseases of the musculoskeletal system and connective tissue",
    "Diseases of the genitourinary system",
    "Pregnancy with abortive outcome",
    "Oedema, proteinuria and hypertensive disorders in pregnancy, childbirth and the puerperium",
    "Other maternal disorders predominantly related to pregnancy",
    "Diabetes mellitus in pregnancy",
    "Maternal infectious and parasitic diseases classifiable elsewhere but complicating pregnancy, childbirth and the puerperium",
    "Certain conditions originating in the perinatal period",
    "Congenital malformations, deformations and chromosomal abnormalities",
    "Symptoms, signs and abnormal clinical and laboratory findings, not elsewhere classified",
    "Injury, poisoning and certain other consequences of external causes",
    "Provisional assignment of new diseases of uncertain etiology or emergency use",
    "Resistance to antimicrobial and antineoplastic drugs",
    "External causes of morbidity and mortality",
    "Factors influencing health status and contact with health services"
]

# Create binary columns for each category 
for cat in categories:
    col_name = cat.replace(" ", "_") \
                  .replace(",", "") \
                  .replace("(", "") \
                  .replace(")", "") \
                  .replace("/", "_") \
                  .replace("-", "_") \
                  .replace("__", "_") 

    df_all_diags = df_all_diags.withColumn(
        col_name,
        when(array_contains(col("DiagDescriptions_ALL"), cat), F.lit(1)).otherwise(F.lit(0))
    )

df_all_diags = df_all_diags.withColumns({
    "Preeclampsia_during_this_pregnancy": when((col("Preeclampsia")==1), 1).otherwise(0),
    "Gestational_Diabetes_during_this_pregnancy": when((col("Diabetes_mellitus_in_pregnancy")==1) | (col("Gestational_Diabetes")==1), 1).otherwise(0),
}).drop(*[
    "Oedema_proteinuria_and_hypertensive_disorders_in_pregnancy_childbirth_and_the_puerperium",
    "Diabetes_mellitus_in_pregnancy",
])


In [0]:
df_preeclampsia = (
    df_all_diags
    .filter(col("Preeclampsia_during_this_pregnancy") == 1)
    .groupBy("UniqPregID")
    .agg(
        first("Preeclampsia_during_this_pregnancy").alias("Preeclampsia_during_this_pregnancy"),
        min("DiagnosisDate").alias("PreeclampsiaDiagnosisDate")
    )
)

df_gestDM = (
    df_all_diags
    .filter(col("Gestational_Diabetes_during_this_pregnancy") == 1)
    .groupBy("UniqPregID")
    .agg(
        first("Gestational_Diabetes_during_this_pregnancy").alias("Gestational_Diabetes_during_this_pregnancy"),
        min("DiagnosisDate").alias("GestDMDiagnosisDate")
    )
)

In [0]:
df_master_final = (
    df_master
    .join(
        df_preeclampsia,
        "UniqPregID",
        "left"
    )
    .join(
        df_gestDM,
        "UniqPregID",
        "left"
    )
)

In [0]:
df_master_final = (
    df_master_final
    .withColumns({
        "Preeclampsia_during_this_pregnancy": when(col("Preeclampsia_during_this_pregnancy") == 1, 1).otherwise(0),
        "Gestational_Diabetes_during_this_pregnancy": when(col("Gestational_Diabetes_during_this_pregnancy") == 1, 1).otherwise(0)
    })
    .withColumns({
        "Preeclampsia_onset": when((col("est_pregnancy_start").isNotNull() & col("PreeclampsiaDiagnosisDate").isNotNull()), F.abs(F.date_diff(col("est_pregnancy_start"), col("PreeclampsiaDiagnosisDate")))).otherwise(None),
        "GestDM_onset": when(col("est_pregnancy_start").isNotNull() & col("GestDMDiagnosisDate").isNotNull(), F.abs(F.date_diff(col("est_pregnancy_start"), col("GestDMDiagnosisDate")))).otherwise(None),
    })
    .withColumns({
        "Preeclampsia_onset": 
            when(((col("Preeclampsia_onset") > 42*7) & col("last_period_date").isNotNull()), F.abs(F.date_diff(col("last_period_date"), col("PreeclampsiaDiagnosisDate")))).when(((col("Preeclampsia_onset") > 42*7) & col("last_period_date").isNull()), None).otherwise(col("Preeclampsia_onset")),
        "GestDM_onset": 
            when(((col("GestDM_onset") > 42*7) & col("last_period_date").isNotNull()), F.abs(F.date_diff(col("last_period_date"), col("GestDMDiagnosisDate")))).when(((col("GestDM_onset") > 42*7) & col("last_period_date").isNull()), None).otherwise(col("GestDM_onset"))
    })
    .withColumns({
        "Preeclampsia_early_onset": when(((col("Preeclampsia_during_this_pregnancy") == 1) & (col("Preeclampsia_onset") < 34*7)), 1).otherwise(0),
        "Preeclampsia_late_onset": when((col("Preeclampsia_during_this_pregnancy") == 0), 0).when(((col("Preeclampsia_during_this_pregnancy") == 1) & (col("Preeclampsia_onset") >= 34*7)), 1).otherwise(None),
        "GestDM_early_onset": when(((col("Gestational_Diabetes_during_this_pregnancy") == 1) & (col("GestDM_onset") < 24*7)), 1).otherwise(0),
        "GestDM_late_onset": when((col("Gestational_Diabetes_during_this_pregnancy") == 0), 0).when(((col("Gestational_Diabetes_during_this_pregnancy") == 1) & (col("GestDM_onset") >= 24*7)), 1).otherwise(None),
    })
)

In [0]:
#%skip
print(f"Preeclampsia prevalence: {100 * df_master_final.filter(col("Preeclampsia_during_this_pregnancy") == 1).count() / df_master_final.filter(col("Preeclampsia_during_this_pregnancy").isNotNull()).count():2f}%")
print(f"Early preeclampsia prevalence: {100 * df_master_final.filter(col("Preeclampsia_early_onset") == 1).count() / df_master_final.filter(col("Preeclampsia_early_onset").isNotNull()).count():2f}%")
print(f"Late preeclampsia prevalence: {100 * df_master_final.filter(col("Preeclampsia_late_onset") == 1).count() / df_master_final.filter(col("Preeclampsia_late_onset").isNotNull()).count():2f}%")

In [0]:
df_preeclampsia_dig = df_master_final.select(["UniqPregID", "Preeclampsia_during_this_pregnancy", "Preeclampsia_early_onset", "Preeclampsia_late_onset", "Preeclampsia_onset"])

In [0]:
write_table_name = "msds_diag_busy_filtered_final_imputed_preeclampsia_diag_stage"

df_preeclampsia_dig.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"dsa_712819_x8g2j.dsa_712819_x8g2j_collab.{write_table_name}")